# Kv2.1 L403A: all-distance sampling breadth at equal depth

This notebook expands the S6-focused sampling analysis to the 546 chain-label-safe intrachain distances shared by the established `all_ok_rmsd_3A_structural_interface_alignment_qc` L403A ensembles: 273 Cα distances and 273 shortest non-hydrogen distances. Raw interchain distances are excluded because Kv2.1 homotetramer chain labels can permute.

It asks whether targeted MSA masking broadens the structural ensemble, whether that effect is detectable in the fixed First100 cohort (20 random seeds × five AlphaFold2 model parameterizations), and how closely First100 results reproduce the complete 100-seed ensemble.

## Statistical design

The principal design varies MSA treatment: the default workflow (vanilla) versus targeted MSA masking. Rows are ordered deterministically by random seed, AlphaFold2 model parameterization, and numbered recycle snapshot; this is an indexed sampling-depth analysis, not timestamped wall-clock chronology. Global IQR ratios describe the complete distributions. For inference, an IQR is calculated within each seed for every distance, making seed—not recycle snapshot—the statistical unit. Vanilla and targeted-masking seed-IQR distributions are compared with two-sided Mann–Whitney tests followed by Benjamini–Hochberg FDR correction within each sampling depth. Distributional separation is additionally summarized by Wasserstein distance between model–seed trajectory medians.

Positive log2 ratios indicate greater ensemble breadth after targeted masking; negative values indicate greater breadth in vanilla predictions. Ensemble breadth is not equivalent to accuracy or experimental relevance.

In [ ]:
from pathlib import Path
import sys, json
import pandas as pd
from IPython.display import display, Image
repo_root=Path.cwd()
if not (repo_root/'shared').is_dir(): repo_root=repo_root.parent
sys.path.insert(0,str(repo_root))
from kv21.run_l403a_all_distance_sampling_analysis import run, TAB, FIG
result=run(); print(json.dumps(result['audit'],indent=2))

## Ensemble-wide summary across sampling depths

In [ ]:
display(pd.read_csv(TAB/'l403a_first100_seed_level_global_breadth_summary.csv'))

In [ ]:
display(pd.read_csv(TAB/'l403a_all_distance_sampling_summary.csv')); display(Image(filename=str(FIG/'all_distance_breadth_fraction_by_depth.png')))

## First100 cohort versus complete 100-seed ensemble

In [ ]:
display(pd.read_csv(TAB/'l403a_all_distance_sampling_stability.csv')); display(Image(filename=str(FIG/'first100_nominal_vs_full_distance_breadth.png')))

## Distribution of breadth effects across the 546 chain-label-safe intrachain distances

In [ ]:
display(Image(filename=str(FIG/'all_distance_iqr_ratio_distributions.png')))

## Strongest concordant broadened and narrowed distances

In [ ]:
display(pd.read_csv(TAB/'top_concordant_distance_breadth_changes.csv')); display(Image(filename=str(FIG/'top_all_distance_breadth_heatmap.png')))

## Machine-readable distance-level statistics

In [ ]:
stats=pd.read_csv(TAB/'l403a_all_distance_sampling_statistics.csv')
display(stats[stats.depth.astype(str).isin(['1000','Full QC'])])
print(f'{len(stats):,} distance-by-depth rows')

## Figure S8 assembly: fixed-budget sampling breadth

This cell assembles the four selected, analysis-generated panels into one editable Figure S8 layout. Each source panel is generated above by `run()` from the saved analysis tables.

- **A:** First100 seed-level breadth, distance-wise breadth, directional test, and retention.
- **B:** Effect magnitude and directional BH evidence across the complete structural-distance panel; every point is one distance.
- **C:** Random-seed saturation from 1,000 subsets per seed budget.
- **D:** First100-versus-full directional-broadening concordance.


In [ ]:
s8_provenance = pd.DataFrame([
    {'panel':'A', 'source function':'figures(...)', 'source data':'overall_seed_breadth + first100_seed_block_distribution_statistics + nominal_trajectory_retention_audit', 'key parameters':'First100 cohort; 20 seeds × 5 model parameterizations; one final retained structure per model–seed trajectory'},
    {'panel':'B', 'source function':'figures(...) evidence scatter', 'source data':'l403a_first100_seed_block_distribution_statistics.csv', 'key parameters':'targeted-masking/vanilla IQR ratio; one-sided seed-block permutation; BH q < 0.05'},
    {'panel':'C', 'source function':'random_seed_saturation(...) → figures(...)', 'source data':'l403a_random_seed_saturation_summary.csv', 'key parameters':'5, 10, 20, 25, 50, 75, 100 seeds; 1,000 random subsets/depth; one final retained structure/model–seed trajectory'},
    {'panel':'D', 'source function':'figures(...) concordance matrix', 'source data':'l403a_first100_vs_full_broadening_concordance.csv', 'key parameters':'directional broader = positive IQR effect and one-sided seed-block BH q < 0.05'},
])
display(s8_provenance)
print('Analysis entry point: kv21.run_l403a_all_distance_sampling_analysis.run()')
print('Figure-generation function: kv21.run_l403a_all_distance_sampling_analysis.figures(...)')


In [ ]:
from PIL import Image as PILImage, ImageChops
import matplotlib.pyplot as plt

panel_paths = {
    'A': FIG/'first100_masked_sampling_breadth_main_summary.png',
    'B': FIG/'first100_all_distance_breadth_evidence_map.png',
    'C': FIG/'random_seed_sampling_efficiency_saturation.png',
    'D': FIG/'first100_vs_full_broadening_concordance_heatmap.png',
}

def trim_white_margin(path, tolerance=8):
    image = PILImage.open(path).convert('RGB')
    background = PILImage.new('RGB', image.size, (255, 255, 255))
    difference = ImageChops.difference(image, background).convert('L')
    difference = difference.point(lambda value: 255 if value > tolerance else 0)
    bounds = difference.getbbox()
    return image.crop(bounds) if bounds else image

panels = {label: trim_white_margin(path) for label, path in panel_paths.items()}

fig = plt.figure(figsize=(22, 17), facecolor='white')
outer = fig.add_gridspec(2, 2, height_ratios=[1, 1], width_ratios=[1, 1], hspace=.035, wspace=.025)
axes = {
    'A': fig.add_subplot(outer[0, 0]),
    'C': fig.add_subplot(outer[0, 1]),
    'B': fig.add_subplot(outer[1, 0]),
    'D': fig.add_subplot(outer[1, 1]),
}

for label, axis in axes.items():
    axis.imshow(panels[label], interpolation='lanczos')
    axis.set_anchor('N')
    axis.axis('off')
    axis.text(-.055, 1.005, label, transform=axis.transAxes, ha='left', va='top',
              fontsize=23, fontweight='bold', color='#111111',
              bbox={'facecolor':'white','edgecolor':'none','pad':1.5})

fig.subplots_adjust(left=.025, right=.995, bottom=.015, top=.995)
composite_png = FIG/'Figure_S8_fixed_budget_sampling_breadth.png'
composite_pdf = FIG/'Figure_S8_fixed_budget_sampling_breadth.pdf'
fig.savefig(composite_png, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(composite_pdf, bbox_inches='tight', facecolor='white')
display(fig)
plt.close(fig)

print('Saved:', composite_png.relative_to(repo_root))
print('Saved:', composite_pdf.relative_to(repo_root))


## Manuscript-facing uncertainty audit

In [ ]:
from analysis.statistics_revision.scripts.run_kv21_sampling_breadth_uncertainty import run as run_uncertainty
run_uncertainty()
revision_tables=repo_root/'analysis'/'statistics_revision'/'tables'
display(pd.read_csv(TAB/'l403a_first100_seed_level_global_breadth_summary.csv'))
display(pd.read_csv(revision_tables/'kv21_s6_masked_vs_vanilla_breadth_bootstrap.csv'))
display(pd.read_csv(revision_tables/'kv21_rmsf_trajectory_block_bootstrap.csv'))
display(pd.read_csv(revision_tables/'kv21_sampling_breadth_manuscript_statistics.csv'))

## Reading guide

- `global_IQR_ratio > 1` means the targeted-masking ensemble is broader than the vanilla ensemble for that distance.
- `seed_IQR_ratio > 1` means the typical within-seed targeted-masking ensemble is broader.
- `seed_breadth_q < 0.05` identifies FDR-significant MSA-treatment differences using seeds as replicates.
- `trajectory_median_W1_A` describes MSA-treatment separation after giving each model–seed trajectory one median observation.
- Agreement between the First100 cohort and complete 100-seed ensemble supports reduced-sampling-depth claims; deterministic seed order is not chronological time.
- These all-distance results establish breadth, not whether broader states are experimentally correct.